# 02 Bad-channel QC and raw events

This notebook assumes that raw BIDS files already exist in the configured
`paths.bids_root`, for example `rawdata/`.

Run `01_raw_bids_import.ipynb` first whenever new recordings or empty-room files
have been added to `sourcedata/`. This notebook then selects recordings from the
fresh raw BIDS dataset, performs manual bad-channel QC, updates `channels.tsv`,
and writes trigger-derived raw BIDS `events.tsv` files.


## Project root and raw BIDS root

This notebook treats the outer project folder as the analysis workspace. The raw BIDS dataset is `config.paths.bids_root`, normally `rawdata/`.

Expected layout:

```text
{{ project_name }}/
  README.md                  # project README
  configs/local.yaml
  notebooks/
  sourcedata/                 # original acquisition exports
  rawdata/                    # raw BIDS dataset root
    README
    dataset_description.json
    participants.tsv
    participants.json
    sub-*
    sub-emptyroom/
  derivatives/
```

Do not place `dataset_description.json`, `participants.tsv`, `participants.json`, or BIDS `sub-*` folders in the outer project root. They belong under `rawdata/`.


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.bids import (
    make_raw_bids_channels_tsv_path,
    make_raw_bids_fif_path,
    read_raw_bids_recording_if_exists,
)
from meeg_pipeline.channels import summarize_channels_tsv
from meeg_pipeline.events import write_bids_events_for_recordings
from meeg_pipeline.qc import (
    bad_channel_candidates_summary_to_dataframe,
    bad_channel_candidates_to_dataframe,
    detect_bad_channel_candidates_maxwell,
    load_bad_channels,
    save_or_load_bad_channels,
)
from meeg_pipeline.workflow import (
    bad_channels_policy_for_step,
    existing_output_policy_for_step,
    iter_recordings,
    recording_label,
    recordings_to_dataframe,
    should_overwrite,
    raw_events_path,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("BIDS_ROOT:", config.paths.bids_root)


In [ ]:
# Check BIDS-root metadata placement.
required_bids_root_files = ["dataset_description.json", "participants.tsv"]
optional_bids_root_files = ["participants.json", "README"]

print("Project root:", PROJECT_ROOT)
print("Raw BIDS root:", config.paths.bids_root)

for filename in required_bids_root_files:
    path = config.paths.bids_root / filename
    print(f"{filename}:", "found" if path.exists() else "MISSING", "-", path)

for filename in optional_bids_root_files:
    path = config.paths.bids_root / filename
    print(f"{filename}:", "found" if path.exists() else "not present", "-", path)


## Selection

Examples:

```python
SUBJECTS = "all"
SUBJECTS = ["0001", "0002"]
SUBJECTS = "0001"

TASKS = "all"
TASKS = ["example"]
TASKS = "example"
```

If the project has no sessions or runs, keep `SESSIONS = None` and `RUNS = None`.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# For interactive QC it is often better to restrict processing explicitly.
# Use None for all selected recordings, or an integer for quick test runs.
MAX_RECORDINGS = None

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

if MAX_RECORDINGS is not None:
    selected_recordings = selected_recordings[:MAX_RECORDINGS]

selected_recordings_table = recordings_to_dataframe(selected_recordings)
selected_recordings_table


## Overwrite policy

Bad-channel decisions are loaded by default when they already exist. Event files
are skipped by default when they already exist.


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "bad_channels",
            "overwrite": should_overwrite("bad_channels", OVERWRITE_STEPS),
            "policy": bad_channels_policy_for_step(
                "bad_channels",
                OVERWRITE_STEPS,
            ),
        },
        {
            "step": "events",
            "overwrite": should_overwrite("events", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "events",
                OVERWRITE_STEPS,
            ),
        },
    ]
)


## Raw BIDS status for selected recordings

This is intentionally a fast path-existence check. It does not open the raw FIF files.
Opening every raw FIF just for a status table is slow and redundant.


In [ ]:
raw_status_rows = []

for recording in selected_recordings:
    raw_path = make_raw_bids_fif_path(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    raw_status_rows.append(
        {
            "recording": recording_label(recording),
            "status": "exists" if raw_path.exists() else "missing_input",
            "message": "" if raw_path.exists() else "Raw BIDS FIF does not exist.",
            "path": raw_path,
        }
    )

raw_status = pd.DataFrame(raw_status_rows)
raw_status


## Channel sidecar summary

This overview reads the small `*_channels.tsv` sidecars instead of opening every raw FIF file.
This keeps the notebook fast even when many recordings exist in `rawdata/`.

The interactive QC cell below still opens the raw FIF for recordings that actually need manual inspection.


In [ ]:
channel_rows = []

for recording in selected_recordings:
    raw_path = make_raw_bids_fif_path(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )
    channels_path = make_raw_bids_channels_tsv_path(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    if not raw_path.exists():
        channel_rows.append(
            {
                "recording": recording_label(recording),
                "status": "missing_input",
                "message": "Raw BIDS FIF does not exist.",
                "n_channels": None,
                "channel_types": "",
                "bad_channels": "",
                "channels_path": channels_path,
            }
        )
        continue

    summary = summarize_channels_tsv(channels_path)
    channel_rows.append(
        {
            "recording": recording_label(recording),
            "status": summary.status,
            "message": summary.message,
            "n_channels": summary.n_channels,
            "channel_types": str(summary.channel_types),
            "bad_channels": ", ".join(summary.bad_channels),
            "channels_path": summary.path,
        }
    )

pd.DataFrame(channel_rows)


## Manual bad-channel QC

This section iterates over all selected recordings.

For each recording:

1. Missing raw files are skipped.
2. Existing bad-channel decisions are loaded unless `OVERWRITE_STEPS` contains `"bad_channels"`.
3. If needed, automatic Maxwell bad-channel candidates are pre-marked.
4. The MNE browser opens with `raw.plot(block=True)`.
5. Final bad-channel decisions are saved and `channels.tsv` is updated.

If you do not want automatic candidates, set `USE_MAXWELL_CANDIDATES = False`.


In [ ]:
RUN_BAD_CHANNEL_QC = True
USE_MAXWELL_CANDIDATES = True
MAXWELL_N_JOBS = 4

# For interactive QC, keep this small when testing.
# Use None to process all selected recordings.
MAX_BAD_CHANNEL_QC_RECORDINGS = None

qc_recordings = selected_recordings
if MAX_BAD_CHANNEL_QC_RECORDINGS is not None:
    qc_recordings = qc_recordings[:MAX_BAD_CHANNEL_QC_RECORDINGS]

if not RUN_BAD_CHANNEL_QC:
    raise RuntimeError(
        "RUN_BAD_CHANNEL_QC is False. Set it to True if you want to open the "
        "interactive bad-channel QC workflow."
    )


bad_channels_policy = bad_channels_policy_for_step(
    "bad_channels",
    OVERWRITE_STEPS,
)

qc_results = []

for index, recording in enumerate(qc_recordings):
    label = recording_label(recording)

    print("=" * 80)
    print(f"Bad-channel QC {index + 1}/{len(qc_recordings)}: {label}")
    print("=" * 80)

    existing = load_bad_channels(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    if existing.status == "loaded" and bad_channels_policy == "load":
        qc_results.append(
            {
                "index": index,
                "recording": label,
                "status": "loaded_existing",
                "message": "Existing bad-channel decision loaded; GUI not opened.",
                "bads": ", ".join(existing.bads),
                "method": existing.method,
                "notes": existing.notes,
                "path": existing.path,
            }
        )
        print("Existing bad-channel decision found; GUI not opened.")
        print(f"Bad channels: {existing.bads}")
        print()
        continue

    raw_result = read_raw_bids_recording_if_exists(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        preload=True,
    )

    if raw_result.raw is None:
        qc_results.append(
            {
                "index": index,
                "recording": label,
                "status": raw_result.status,
                "message": raw_result.message,
                "bads": "",
                "method": "",
                "notes": "",
                "path": raw_result.path,
            }
        )
        print(raw_result.message)
        print()
        continue

    raw_qc = raw_result.raw

    if USE_MAXWELL_CANDIDATES:
        candidates = detect_bad_channel_candidates_maxwell(
            raw_qc,
            n_jobs=MAXWELL_N_JOBS,
        )
        display(bad_channel_candidates_summary_to_dataframe(candidates))
        display(bad_channel_candidates_to_dataframe(candidates))
        raw_qc.info["bads"] = candidates.combined

        method = "manual_mne_gui_with_maxwell_candidates"
        notes = (
            "Automatic Maxwell bad-channel candidates were pre-marked and "
            "manually reviewed with raw.plot(block=True)."
        )
    else:
        method = "manual_mne_gui"
        notes = "Marked interactively with raw.plot(block=True)."

    raw_qc.plot(block=True)

    result = save_or_load_bad_channels(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        bads=raw_qc.info["bads"],
        method=method,
        notes=notes,
        on_existing=bad_channels_policy,
        update_channels_tsv=True,
    )

    qc_results.append(
        {
            "index": index,
            "recording": label,
            "status": result.status,
            "message": result.message,
            "bads": ", ".join(result.bads),
            "method": result.method,
            "notes": result.notes,
            "path": result.path,
        }
    )

qc_results_table = pd.DataFrame(qc_results)
qc_results_table


## Write events.tsv for selected recordings

Existing `events.tsv` files are skipped by default. Missing raw files are reported as status rows.


In [ ]:
events_policy = existing_output_policy_for_step(
    "events",
    OVERWRITE_STEPS,
)

write_results = write_bids_events_for_recordings(
    config,
    selected_recordings,
    on_existing=events_policy,
)

pd.DataFrame(
    [
        {
            "status": result.status,
            "message": result.message,
            "n_events": result.n_events,
            "unique_ids": ", ".join(map(str, result.unique_ids or [])),
            "output_path": result.output_path,
        }
        for result in write_results
    ]
)

## Events status

This checks expected events files for all selected recordings.


In [ ]:
event_status_rows = []

for recording in selected_recordings:
    events_path = raw_events_path(config, recording)

    event_status_rows.append(
        {
            "recording": recording_label(recording),
            "events_exists": events_path.exists(),
            "events_path": str(events_path),
        }
    )

pd.DataFrame(event_status_rows)